In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Alignment Summary Table for BrainVital8
=========================================
Extract invariance alignment parameters from each cohort vs UKB anchor:
  - alpha_0 (location), psi_0 (scale)
  - R² and sqrt(U²) for loadings and intercepts

Output: alignment_summary_paper.csv
"""

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter


# ========================= Configuration =========================

COHORT_FILES = {
    "KLoSA":  "./data/KLOSA/KLOSA_UKB_alignment_params.csv",
    "CHARLS": "./data/CHARLS/CHARLS_UKB_alignment_params.csv",
    "HRS":    "./data/HRS/HRS_UKB_alignment_params.csv",
    "ELSA":   "./data/ELSA/ELSA_UKB_alignment_params.csv",
    "SHARE":  "./data/SHARE/SHARE_UKB_alignment_params.csv",
}

COHORT_ALIGNMENT_DATA = {
    "KLoSA": {
        "bv8_file": "./data/KLOSA/KLOSA_BrainVital8_domains_for_alignment.csv",
    },
    "CHARLS": {
        "bv8_file": "./data/CHARLS/CHARLS_BrainVital8_domains_for_alignment.csv",
    },
    "HRS": {
        "bv8_file": "./data/HRS/HRS_BrainVital8_domains_for_alignment.csv",
    },
    "ELSA": {
        "bv8_file": "./data/ELSA/ELSA_BrainVital8_domains_for_alignment.csv",
    },
    "SHARE": {
        "bv8_file": "./data/SHARE/SHARE_BrainVital8_domains_for_alignment.csv",
    },
}

UKB_ANCHOR = "./data/BEST_BrainVital8_alignment.csv"
OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ALIGN_DOMAINS = [
    "respiratory_comp", "glucose_comp", "sleep_comp", "grip_comp",
    "activity_comp", "SDOH_comp", "emotional_comp", "smoke_comp",
]


# ========================= Step 1: Load UKB Anchor =========================

print("=" * 80)
print("Step 1: Load UKB anchor")
print("=" * 80)

if not os.path.exists(UKB_ANCHOR):
    raise FileNotFoundError(f"UKB anchor not found: {UKB_ANCHOR}")

ukb_align = pd.read_csv(UKB_ANCHOR)
print(f"  UKB anchor: {ukb_align.shape}")

missing = [c for c in ALIGN_DOMAINS if c not in ukb_align.columns]
if missing:
    raise ValueError(f"UKB anchor missing columns: {missing}")


# ========================= Step 2: Run Alignment =========================

print("\n" + "=" * 80)
print("Step 2: Run alignment for each cohort")
print("=" * 80)

sirt_pkg = importr("sirt")

# Push UKB data to R
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["ukb_align_r"] = ro.conversion.get_conversion().py2rpy(
        ukb_align[ALIGN_DOMAINS])

results = []

for cohort_name, cfg in COHORT_ALIGNMENT_DATA.items():
    print(f"\n  [{cohort_name}]")

    if not os.path.exists(cfg["bv8_file"]):
        print(f"    Warning: file not found: {cfg['bv8_file']}")
        continue

    cohort_data = pd.read_csv(cfg["bv8_file"])
    missing_c = [c for c in ALIGN_DOMAINS if c not in cohort_data.columns]
    if missing_c:
        print(f"    Warning: missing columns: {missing_c}")
        continue

    print(f"    Data: {len(cohort_data):,} rows")

    # Push cohort data to R
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv["cohort_align_r"] = ro.conversion.get_conversion().py2rpy(
            cohort_data[ALIGN_DOMAINS])

    ro.globalenv["cohort_label"] = cohort_name

    try:
        # Run invariance alignment via sirt
        ro.r('''
        domains <- c("respiratory_comp","glucose_comp","sleep_comp","grip_comp",
                     "activity_comp","SDOH_comp","emotional_comp","smoke_comp")
        combined <- rbind(ukb_align_r[, domains], cohort_align_r[, domains])
        grp <- c(rep("UKB", nrow(ukb_align_r)),
                 rep(cohort_label, nrow(cohort_align_r)))

        cfg <- sirt::invariance_alignment_cfa_config(
            dat=combined, group=grp, model="2PM", verbose=FALSE)
        aln <- sirt::invariance.alignment(
            lambda=cfg$lambda, nu=cfg$nu, optimizer="nlminb")

        # es.invariance matrix: row 1=R², row 2=sqrt(U²); col 1=loadings, col 2=intercepts
        r2_load   <- aln$es.invariance[1,1]
        r2_int    <- aln$es.invariance[1,2]
        u2_load   <- aln$es.invariance[2,1]
        u2_int    <- aln$es.invariance[2,2]

        # Linking parameters: row 1=UKB, row 2=cohort
        alpha0_cohort <- aln$pars[2, 1]
        psi0_cohort   <- aln$pars[2, 2]
        ''')

        # Read back to Python
        r2_load = float(ro.r("r2_load")[0])
        r2_int = float(ro.r("r2_int")[0])
        u2_load = float(ro.r("u2_load")[0])
        u2_int = float(ro.r("u2_int")[0])
        alpha0 = float(ro.r("alpha0_cohort")[0])
        psi0 = float(ro.r("psi0_cohort")[0])

        print(f"    R2(load)={r2_load:.4f}  R2(int)={r2_int:.4f}  "
              f"sqrtU2(load)={u2_load:.4f}  sqrtU2(int)={u2_int:.4f}")
        print(f"    alpha0={alpha0:.4f}  psi0={psi0:.4f}")

        # Verdict
        r2_load_verdict = (
            "Strong" if r2_load >= 0.85 else
            "Acceptable" if r2_load >= 0.75 else
            "Marginal" if r2_load >= 0.60 else
            "Poor"
        )
        psi0_verdict = (
            "Good" if 0.5 <= psi0 <= 2 else
            "Moderate" if 0.3 <= psi0 <= 5 else
            "Poor"
        )

        results.append({
            "Cohort": cohort_name,
            "N": len(cohort_data),
            "alpha_0": round(alpha0, 4),
            "psi_0": round(psi0, 4),
            "R2_loadings": round(r2_load, 4),
            "R2_intercepts": round(r2_int, 4),
            "sqrtU2_loadings": round(u2_load, 4),
            "sqrtU2_intercepts": round(u2_int, 4),
            "R2_load_verdict": r2_load_verdict,
            "psi0_verdict": psi0_verdict,
        })

    except Exception as e:
        print(f"    Error: {e}")
        continue


# ========================= Step 3: Export Summary Table =========================

print("\n" + "=" * 80)
print("Step 3: Export summary table")
print("=" * 80)

if len(results) == 0:
    print("  No cohort results available")
else:
    df_result = pd.DataFrame(results)

    # Add UKB anchor row
    ukb_row = pd.DataFrame([{
        "Cohort": "UKB (anchor)",
        "N": len(ukb_align),
        "alpha_0": 0.0000,
        "psi_0": 1.0000,
        "R2_loadings": np.nan,
        "R2_intercepts": np.nan,
        "sqrtU2_loadings": np.nan,
        "sqrtU2_intercepts": np.nan,
        "R2_load_verdict": "Anchor",
        "psi0_verdict": "Anchor",
    }])
    df_full = pd.concat([ukb_row, df_result], ignore_index=True)

    # Export CSV
    out_path = os.path.join(OUTPUT_DIR, "alignment_summary_paper.csv")
    df_full.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n  Exported: {out_path}")

    # Print summary
    print(f"\n  Summary table:")
    print(df_full.to_string(index=False))

print("\n" + "=" * 80)
print("Done")
print("=" * 80)